# API de Filmes - Google Colab

## 1. Instalar bibliotecas

In [1]:
!pip install -q scikit-learn fastapi pandas uvicorn jinja2 nest-asyncio pyngrok

## 2. Criar os dados

In [2]:
import pandas as pd

dados = {
    "id": range(1, 11),
    "titulo": [
        "Interestelar", "O Poderoso Chefão", "Matrix", "Clube da Luta",
        "Os Vingadores", "Titanic", "Avatar", "Corra!", "Homem-Aranha", "Parasita"
    ],
    "genero": [
        "Ficção Científica", "Drama", "Ação", "Drama", "Ação",
        "Romance", "Ficção Científica", "Terror", "Ação", "Drama"
    ],
    "ano": [2014, 1972, 1999, 1999, 2012, 1997, 2009, 2017, 2002, 2019],
    "duracao": [169, 175, 136, 139, 143, 195, 162, 104, 121, 132],
    "nota": [8.7, 9.2, 8.7, 8.8, 8.0, 7.9, 7.8, 7.7, 7.4, 8.5]
}

df = pd.DataFrame(dados)
df.to_csv("filmes.csv", index=False)
df

,id,titulo,genero,ano,duracao,nota
0,1,Interestelar,Ficção Científica,2014,169,8.7
1,2,O Poderoso Chefão,Drama,1972,175,9.2
2,3,Matrix,Ação,1999,136,8.7
3,4,Clube da Luta,Drama,1999,139,8.8
4,5,Os Vingadores,Ação,2012,143,8.0
5,6,Titanic,Romance,1997,195,7.9
6,7,Avatar,Ficção Científica,2009,162,7.8
7,8,Corra!,Terror,2017,104,7.7
8,9,Homem-Aranha,Ação,2002,121,7.4
9,10,Parasita,Drama,2019,132,8.5


## 3. Criar a API

In [3]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(
    title="API de Filmes",
    description="API para consultar e cadastrar filmes",
    version="1.0.0"
)

df = pd.read_csv("filmes.csv")

class Filme(BaseModel):
    titulo: str
    genero: str
    ano: int
    duracao: int
    nota: float

@app.get("/")
def inicio():
    return {"message": "API de Filmes funcionando!"}

@app.get("/filmes")
def listar_filmes():
    return df.to_dict(orient="records")

@app.get("/filmes/{filme_id}")
def buscar_filme(filme_id: int):
    filme = df[df["id"] == filme_id]
    if filme.empty:
        raise HTTPException(status_code=404, detail="Filme não encontrado")
    return filme.iloc[0].to_dict()

@app.get("/filmes/genero/{genero}")
def buscar_por_genero(genero: str):
    filmes = df[df["genero"].str.lower() == genero.lower()]
    if filmes.empty:
        raise HTTPException(status_code=404, detail="Nenhum filme encontrado")
    return filmes.to_dict(orient="records")

@app.post("/filmes", status_code=201)
def criar_filme(filme: Filme):
    global df
    novo_id = int(df["id"].max()) + 1
    novo_filme = {
        "id": novo_id,
        "titulo": filme.titulo,
        "genero": filme.genero,
        "ano": filme.ano,
        "duracao": filme.duracao,
        "nota": filme.nota
    }
    df.loc[len(df)] = novo_filme
    return novo_filme

@app.delete("/filmes/{filme_id}")
def deletar_filme(filme_id: int):
    global df
    if filme_id not in df["id"].values:
        raise HTTPException(status_code=404, detail="Filme não encontrado")
    df = df[df["id"] != filme_id]
    return {"message": "Filme removido com sucesso"}

## 4. Rodar a API

In [4]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

def iniciar_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=iniciar_api)
thread.start()

## 5. Criar link público

In [ ]:
from pyngrok import ngrok

# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# Replace "YOUR_AUTH_TOKEN" with your actual ngrok authtoken
# ngrok.set_auth_token("YOUR_AUTH_TOKEN")

public_url = ngrok.connect(8000)
print("API:", public_url)
print("Swagger:", public_url + "/docs")